In [ ]:
# Load dataset
import pandas as pd
import numpy as np

# --- LOADING AND CLEANING ---
file_path = "ALLFLOWMETER_HIKARI2021.csv"
df = pd.read_csv(file_path)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
df_filtrado = df.iloc[:, 7:].copy()  # Remove IPs, ports, ID

df.iloc[:, 7:].copy()  # Remove IPs, ports, ID

In [ ]:
%%writefile ids_engine_blackbox2.py
import os
import time
import random
import gc
import numpy as np
import pandas as pd
import multiprocessing as mp
import tensorflow as tf

from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE


# ------------------------------------------------------------
# MEMORY PROTECTION
# ------------------------------------------------------------
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"


# ------------------------------------------------------------
# GENERAL SETTINGS
# ------------------------------------------------------------
N_RUNS = 30
EPSILONS = [0.001, 0.005, 0.01, 0.02, 0.05]
BATCH_SIZE = 32
ADV_BATCH_SIZE = 64
EPOCHS = 100
THRESHOLD = 0.5


# ------------------------------------------------------------
# HIKARI STRICT PLAUSIBILITY MASK
# ------------------------------------------------------------
# The HIKARI dataframe is expected in one of these layouts:
# 1) complete HIKARI layout described in the LaTeX document:
#    uid, originh, originp, responh, responp,
#    79 numerical traffic features, traffic_category, Label;
# 2) filtered layout:
#    79 numerical traffic features, traffic_category, Label;
# 3) complete layout with extra index/metadata columns.
#
# The plausibility mask is applied only to the 79 numerical traffic features.
# traffic_category and Label are responses / ground truth and must never be
# perturbed.
#
# Convention:
# - 1.0 means perturbable;
# - 0.0 means blocked.
#
# Strategy:
# - deny by default;
# - enable only the temporal/rate features justified by domain constraints.
#
# Rationale:
# - timing and rate features can be modified by delaying, spacing, or changing
#   the rhythm of packets without directly changing the malicious payload;
# - identifiers, IPs, ports, labels, TCP flags, headers, window sizes, packet
#   counters, bulk, subflow, and raw payload statistics are blocked in this
#   strict mask because independent perturbations can violate protocol rules,
#   feature interdependencies, or semantic consistency.
#
# This mask is intended for both white-box and black-box evasion experiments.

HIKARI_RESPONSE_COLUMNS = ["traffic_category", "Label"]

HIKARI_METADATA_COLUMNS = [
    "uid",
    "originh",
    "originp",
    "responh",
    "responp",
]

HIKARI_ALLOWED_FEATURES = [
    # Flow duration and packet-rate features.
    "flow_duration",
    "fwd_pkts_per_sec",
    "bwd_pkts_per_sec",
    "flow_pkts_per_sec",

    # Forward inter-arrival time statistics.
    "fwd_iat.min",
    "fwd_iat.max",
    "fwd_iat.tot",
    "fwd_iat.avg",
    "fwd_iat.std",

    # Backward inter-arrival time statistics.
    "bwd_iat.min",
    "bwd_iat.max",
    "bwd_iat.tot",
    "bwd_iat.avg",
    "bwd_iat.std",

    # Bidirectional flow inter-arrival time statistics.
    "flow_iat.min",
    "flow_iat.max",
    "flow_iat.tot",
    "flow_iat.avg",
    "flow_iat.std",

    # Payload transmission rate.
    "payload_bytes_per_second",

    # Active-time statistics.
    "active.min",
    "active.max",
    "active.tot",
    "active.avg",
    "active.std",

    # Idle-time statistics.
    "idle.min",
    "idle.max",
    "idle.tot",
    "idle.avg",
    "idle.std",
]


def build_hikari_plausibility_mask(feature_cols):
    """
    Builds the strict temporal/rate HIKARI plausibility mask from feature names.

    The mask uses the following convention:
    - 1.0 means that the feature is perturbable;
    - 0.0 means that the feature is blocked.

    This function expects only the 79 numerical traffic features as input.
    traffic_category and Label must not be included because they are response
    variables, not model features.

    The adopted strategy is deny-by-default:
    - all features are blocked initially;
    - only HIKARI_ALLOWED_FEATURES are enabled.

    This prevents the optimizer from directly modifying:
    - labels and traffic categories;
    - identifiers, IPs, and ports;
    - TCP flags and header/window fields;
    - packet counters, bulk/subflow fields, and raw payload statistics;
    - derived volume fields whose independent modification would break feature
      interdependencies.

    The enabled subset follows the plausibility assumption that timing/rate
    features can be changed through packet scheduling while preserving the
    malicious semantics of the original flow.
    """
    feature_cols = list(feature_cols)

    if len(feature_cols) != 79:
        raise ValueError(
            f"The HIKARI plausibility mask expects 79 numerical features, "
            f"but received {len(feature_cols)} columns."
        )

    leaked_responses = [
        col for col in HIKARI_RESPONSE_COLUMNS
        if col in feature_cols
    ]

    if leaked_responses:
        raise ValueError(
            "Response columns were found among the input features: "
            f"{leaked_responses}. traffic_category and Label must not be part "
            "of the plausibility mask."
        )

    mask_series = pd.Series(0.0, index=feature_cols, dtype="float32")

    missing_features = [
        feature for feature in HIKARI_ALLOWED_FEATURES
        if feature not in mask_series.index
    ]

    if missing_features:
        raise ValueError(
            "The following HIKARI allowed features were not found: "
            f"{missing_features}"
        )

    mask_series.loc[HIKARI_ALLOWED_FEATURES] = 1.0

    return mask_series.values.astype("float32"), mask_series


def infer_hikari_feature_target_columns(df):
    """
    Infers the HIKARI feature columns and binary target column.

    This function accepts:
    1. the complete HIKARI dataframe described in the LaTeX document:
       5 metadata columns + 79 features + traffic_category + Label;
    2. the filtered dataframe:
       79 features + traffic_category + Label;
    3. the complete dataframe with extra index/metadata columns, as long as
       the 79 feature columns can be inferred.

    Returns
    -------
    X : pandas.DataFrame
        Dataframe containing only the 79 numerical feature columns.
    y : pandas.Series
        Binary target column.
    feature_cols : list
        Names of the 79 feature columns.
    """
    if "Label" in df.columns:
        y = df["Label"].astype("int32").copy()
    else:
        y = df.iloc[:, -1].astype("int32").copy()

    candidate_column_groups = []

    # Candidate based on explicit column names:
    # remove known metadata and response columns. This matches the LaTeX
    # description: uid/origin/respon columns are not attack features.
    if set(HIKARI_RESPONSE_COLUMNS).issubset(set(df.columns)):
        candidate_column_groups.append(
            [
                col for col in df.columns
                if col not in HIKARI_RESPONSE_COLUMNS
                and col not in HIKARI_METADATA_COLUMNS
            ]
        )

    # Candidate for the LaTeX HIKARI layout:
    # 5 metadata columns + 79 features + 2 response columns = 86 columns.
    if df.shape[1] >= 86:
        candidate_column_groups.append(list(df.columns[5:-2]))

    # Candidate for layouts with two extra index/metadata columns:
    # 7 metadata/index columns + 79 features + 2 response columns = 88 columns.
    if df.shape[1] >= 88:
        candidate_column_groups.append(list(df.columns[7:-2]))

    # Candidate for the already filtered dataframe:
    # 79 features + traffic_category + Label.
    if df.shape[1] >= 81:
        candidate_column_groups.append(list(df.columns[:-2]))

    # Fallback for generic input using the original notebook convention.
    candidate_column_groups.append(list(df.columns[:-2]))

    for feature_cols in candidate_column_groups:
        feature_set = set(feature_cols)
        has_no_response_leakage = not any(
            col in feature_set for col in HIKARI_RESPONSE_COLUMNS
        )
        has_no_metadata_leakage = not any(
            col in feature_set for col in HIKARI_METADATA_COLUMNS
        )

        if (
            len(feature_cols) == 79
            and has_no_response_leakage
            and has_no_metadata_leakage
            and set(HIKARI_ALLOWED_FEATURES).issubset(feature_set)
        ):
            X = df.loc[:, feature_cols].copy()
            return X, y, feature_cols

    raise ValueError(
        "Could not infer the 79 HIKARI numerical feature columns. "
        "Expected either the complete dataframe with metadata, 79 features, "
        "traffic_category and Label, or the filtered dataframe with "
        "79 features plus traffic_category and Label."
    )


def summarize_hikari_plausibility_mask(mask_series):
    """
    Prints a concise summary of the HIKARI plausibility mask.
    """
    n_total = int(mask_series.shape[0])
    n_perturbable = int(mask_series.sum())
    n_blocked = int((mask_series == 0.0).sum())

    print("\n" + "=" * 70)
    print("HIKARI STRICT TEMPORAL/RATE PLAUSIBILITY MASK")
    print("=" * 70)
    print(f"Total features: {n_total}")
    print(f"Perturbable features: {n_perturbable}")
    print(f"Blocked features: {n_blocked}")

    print("\nPerturbable features:")
    for feature_name in mask_series[mask_series == 1.0].index:
        print(f"  - {feature_name}")

    print("\nBlocked features:")
    for feature_name in mask_series[mask_series == 0.0].index:
        print(f"  - {feature_name}")

    print("=" * 70)


def configure_gpu():
    """
    Configures dynamic GPU memory growth.
    """
    gpus = tf.config.list_physical_devices("GPU")

    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass


def set_seed(seed):
    """
    Sets seeds for reproducibility.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


# ------------------------------------------------------------
# TARGET MODEL M1
# ------------------------------------------------------------
def build_mtl_model_M1(input_shape):
    """
    Target model M1: CNN (3 layers) + ECA + Transformer infrastructure.
    """
    inputs = layers.Input(shape=input_shape)

    def eca_block(input_tensor):
        channels = input_tensor.shape[-1]

        squeeze = layers.GlobalAveragePooling2D()(input_tensor)
        squeeze = layers.Reshape((1, 1, channels))(squeeze)

        k_size = max(3, int(abs((np.log2(channels) + 1) / 2 + 0.5)))

        squeeze = layers.Conv2D(
            1,
            kernel_size=(1, k_size),
            padding="same",
            activation="sigmoid",
            use_bias=False,
        )(squeeze)

        return layers.Multiply()([input_tensor, squeeze])

    def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0.3):
        x = layers.LayerNormalization(epsilon=1e-6)(inputs)

        x = layers.MultiHeadAttention(
            key_dim=head_size,
            num_heads=num_heads,
            dropout=dropout,
        )(x, x)

        x = layers.Dropout(dropout)(x)
        res = x + inputs

        x = layers.LayerNormalization(epsilon=1e-6)(res)
        x = layers.Dense(ff_dim, activation="relu")(x)
        x = layers.Dropout(dropout)(x)
        x = layers.Dense(inputs.shape[-1])(x)

        return x + res

    # Convolutional Block 1
    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(inputs)
    x = layers.MaxPooling2D(pool_size=(2, 2), padding="same")(x)
    x = eca_block(x)

    # Convolutional Block 2
    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2), padding="same")(x)
    x = eca_block(x)

    # Convolutional Block 3 (Expansion to the paper's M1 pattern)
    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2), padding="same")(x)
    x = eca_block(x)

    # Transition to the Temporal Stage (Transformer)
    x = layers.Reshape((-1, x.shape[-1]))(x)
    x = transformer_encoder(
        x,
        head_size=128,
        num_heads=4,
        ff_dim=256,
        dropout=0.3,
    )

    # Final classifier
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu", kernel_regularizer=l2(1e-2))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=l2(1e-2))(x)

    output = layers.Dense(
        1,
        activation="sigmoid",
        dtype="float32",
        name="binary_output",
    )(x)

    model = Model(inputs=inputs, outputs=output)

    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss=BinaryCrossentropy(),
        metrics=["accuracy", "Precision", "Recall", "AUC"],
    )

    return model


# ------------------------------------------------------------
# SURROGATE MODEL
# ------------------------------------------------------------
def build_surrogate_model(input_shape):
    """
    Surrogate model used in the query-based surrogate black-box scenario.
    """
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(inputs)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    squeeze = layers.GlobalAveragePooling2D()(x)
    excitation = layers.Dense(128 // 4, activation="relu")(squeeze)
    excitation = layers.Dense(128, activation="sigmoid")(excitation)
    excitation = layers.Reshape((1, 1, 128))(excitation)

    x = layers.Multiply()([x, excitation])
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)

    output = layers.Dense(1, activation="sigmoid", dtype="float32")(x)

    model = Model(inputs=inputs, outputs=output)

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss=BinaryCrossentropy(),
        metrics=["accuracy"],
    )

    return model


# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------
def get_metrics(y_true, y_pred_proba, threshold=0.5):
    """
    Computes binary classification metrics.
    """
    y_true = np.asarray(y_true).reshape(-1)
    y_pred_proba = np.asarray(y_pred_proba).reshape(-1)

    y_pred = (y_pred_proba > threshold).astype("int32")

    return {
        "Acc": accuracy_score(y_true, y_pred),
        "Prec": precision_score(y_true, y_pred, zero_division=0),
        "Rec": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "AUC": roc_auc_score(y_true, y_pred_proba),
    }


def calculate_ASR_I(y_true, y_clean_proba, y_adv_proba, threshold=0.5):
    """
    Computes the Attack Success Rate (ASR_I).

    ASR_I considers only malicious samples that were correctly
    detected as malicious before the attack.
    """
    y_true = np.asarray(y_true).reshape(-1)

    y_clean_pred = (np.asarray(y_clean_proba).reshape(-1) > threshold).astype("int32")
    y_adv_pred = (np.asarray(y_adv_proba).reshape(-1) > threshold).astype("int32")

    originally_malicious = (y_true == 1) & (y_clean_pred == 1)
    successfully_evaded = originally_malicious & (y_adv_pred == 0)

    denominator = np.sum(originally_malicious)

    return np.sum(successfully_evaded) / denominator if denominator > 0 else 0.0


# ------------------------------------------------------------
# DATA PREPARATION
# ------------------------------------------------------------
def reshape_to_2d(data, size):
    """
    Applies padding up to the perfect square and reshapes for Conv2D input.
    """
    pad_size = size**2 - data.shape[1]

    padded = np.pad(
        data,
        pad_width=((0, 0), (0, pad_size)),
        mode="constant",
    )

    return padded.reshape(-1, size, size, 1).astype("float32")


def prepare_splits(X, y, seed):
    """
    Splits the data into training, validation, and test sets.
    Applies SMOTE only to the training set.
    Fits MinMaxScaler only on the training set.
    """
    X_train, X_temp, y_train, y_temp = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=seed,
        stratify=y,
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.50,
        random_state=seed,
        stratify=y_temp,
    )

    smote = SMOTE(random_state=seed)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

    scaler = MinMaxScaler()

    X_train_scaled = scaler.fit_transform(X_train_res).astype("float32")
    X_val_scaled = scaler.transform(X_val).astype("float32")
    X_test_scaled = scaler.transform(X_test).astype("float32")

    size = int(np.ceil(np.sqrt(X_train_scaled.shape[1])))

    return (
        reshape_to_2d(X_train_scaled, size),
        reshape_to_2d(X_val_scaled, size),
        reshape_to_2d(X_test_scaled, size),
        np.asarray(y_train_res).astype("float32"),
        np.asarray(y_val).astype("float32"),
        np.asarray(y_test).astype("float32"),
        size,
        scaler,
    )


def make_tf_dataset(X_data, y_data, batch_size, shuffle=False, seed=42):
    """
    Creates a batched tf.data.Dataset.
    """
    X_data = X_data.astype("float32")
    y_data = np.asarray(y_data).astype("float32").reshape(-1, 1)

    with tf.device("/CPU:0"):
        dataset = tf.data.Dataset.from_tensor_slices((X_data, y_data))

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=min(len(X_data), 10000),
            seed=seed,
            reshuffle_each_iteration=True,
        )

    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)


# ------------------------------------------------------------
# ADVERSARIAL ATTACKS ON THE SURROGATE
# ------------------------------------------------------------
def fgsm_attack_masked(model, x, y, epsilon, feature_mask_2d, batch_size=256):
    """
    Generates the FGSM attack on the surrogate with a feature mask.
    """
    x_adv_list = []
    y_array = np.asarray(y).reshape(-1).astype("float32")

    mask_tensor = tf.convert_to_tensor(feature_mask_2d, dtype=tf.float32)

    @tf.function
    def fgsm_step(x_batch_tensor, y_batch_tensor):
        with tf.GradientTape() as tape:
            tape.watch(x_batch_tensor)

            y_pred = model(x_batch_tensor, training=False)

            y_pred_safe = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
            logits = tf.math.log(y_pred_safe / (1.0 - y_pred_safe))

            loss = tf.nn.sigmoid_cross_entropy_with_logits(
                labels=y_batch_tensor,
                logits=logits,
            )

        gradient = tape.gradient(loss, x_batch_tensor)
        masked_gradient = gradient * mask_tensor

        x_adv = x_batch_tensor + epsilon * tf.sign(masked_gradient)

        return tf.clip_by_value(x_adv, 0.0, 1.0)

    for start in range(0, len(x), batch_size):
        end = start + batch_size

        x_tensor = tf.convert_to_tensor(
            x[start:end].astype("float32"),
            dtype=tf.float32,
        )

        y_tensor = tf.convert_to_tensor(
            y_array[start:end].reshape(-1, 1),
            dtype=tf.float32,
        )

        x_adv_list.append(fgsm_step(x_tensor, y_tensor).numpy())
        gc.collect()

    return np.concatenate(x_adv_list, axis=0).astype("float32")


def pgd_attack_masked(model, x, y, epsilon, alpha, steps, feature_mask_2d, batch_size=256):
    """
    Generates the PGD attack on the surrogate with a feature mask.
    """
    x_adv_list = []
    y_array = np.asarray(y).reshape(-1).astype("float32")

    mask_tensor = tf.convert_to_tensor(feature_mask_2d, dtype=tf.float32)

    @tf.function
    def pgd_step(x_adv_tensor, x_orig_tensor, y_tensor):
        with tf.GradientTape() as tape:
            tape.watch(x_adv_tensor)

            y_pred = model(x_adv_tensor, training=False)

            y_pred_safe = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
            logits = tf.math.log(y_pred_safe / (1.0 - y_pred_safe))

            loss = tf.nn.sigmoid_cross_entropy_with_logits(
                labels=y_tensor,
                logits=logits,
            )

        gradient = tape.gradient(loss, x_adv_tensor)
        masked_gradient = gradient * mask_tensor

        x_adv_tensor = x_adv_tensor + alpha * tf.sign(masked_gradient)

        perturbation = tf.clip_by_value(
            x_adv_tensor - x_orig_tensor,
            -epsilon,
            epsilon,
        )

        return tf.clip_by_value(x_orig_tensor + perturbation, 0.0, 1.0)

    for start in range(0, len(x), batch_size):
        end = start + batch_size

        x_original = tf.convert_to_tensor(
            x[start:end].astype("float32"),
            dtype=tf.float32,
        )

        x_adv = tf.identity(x_original)

        y_tensor = tf.convert_to_tensor(
            y_array[start:end].reshape(-1, 1),
            dtype=tf.float32,
        )

        for _ in range(steps):
            x_adv = pgd_step(x_adv, x_original, y_tensor)

        x_adv_list.append(x_adv.numpy())
        gc.collect()

    return np.concatenate(x_adv_list, axis=0).astype("float32")


# ------------------------------------------------------------
# BLACK-BOX ADVERSARIAL EVALUATION
# ------------------------------------------------------------
def evaluate_adversarial_blackbox(
    target_model,
    surrogate_model,
    X_test_2d,
    y_test,
    epsilons,
    y_clean_proba,
    feature_mask_2d,
):
    """
    Generates adversarial examples on the surrogate and evaluates the transfer
    of these examples to the target model, accounting for generation and
    inference time.

    Ground-truth test labels are not used to generate the attacks.
    They are used only for final metric computation. FGSM and PGD are generated
    using pseudo-labels predicted by the surrogate model.
    """
    results = []

    # In the black-box setting, ground-truth test labels are used only
    # for evaluation. Attack generation uses pseudo-labels produced by
    # the surrogate model itself.
    y_test_pseudo_proba = surrogate_model.predict(
        X_test_2d.astype("float32"),
        batch_size=ADV_BATCH_SIZE,
        verbose=0,
    )

    y_test_pseudo = (
        y_test_pseudo_proba.reshape(-1) > THRESHOLD
    ).astype("float32")

    # Variables to accumulate the total time (all epsilons)
    total_time_fgsm_gen = 0.0
    total_time_fgsm_inf = 0.0
    total_time_pgd_gen = 0.0
    total_time_pgd_inf = 0.0

    for eps in epsilons:
        # --- FGSM ---
        start_fgsm_gen = time.perf_counter()
        X_fgsm = fgsm_attack_masked(
            surrogate_model,
            X_test_2d,
            y_test_pseudo,
            eps,
            feature_mask_2d,
            ADV_BATCH_SIZE,
        )
        end_fgsm_gen = time.perf_counter()
        t_fgsm_gen = end_fgsm_gen - start_fgsm_gen
        total_time_fgsm_gen += t_fgsm_gen

        # --- PGD ---
        start_pgd_gen = time.perf_counter()
        X_pgd = pgd_attack_masked(
            surrogate_model,
            X_test_2d,
            y_test_pseudo,
            eps,
            eps / 4,
            10,
            feature_mask_2d,
            ADV_BATCH_SIZE,
        )
        end_pgd_gen = time.perf_counter()
        t_pgd_gen = end_pgd_gen - start_pgd_gen
        total_time_pgd_gen += t_pgd_gen

        # --- FGSM INFERENCE ON THE TARGET ---
        start_fgsm_inf = time.perf_counter()
        y_fgsm_proba = target_model.predict(
            X_fgsm,
            batch_size=ADV_BATCH_SIZE,
            verbose=0,
        )
        end_fgsm_inf = time.perf_counter()
        t_fgsm_inf = end_fgsm_inf - start_fgsm_inf
        total_time_fgsm_inf += t_fgsm_inf

        # --- PGD INFERENCE ON THE TARGET ---
        start_pgd_inf = time.perf_counter()
        y_pgd_proba = target_model.predict(
            X_pgd,
            batch_size=ADV_BATCH_SIZE,
            verbose=0,
        )
        end_pgd_inf = time.perf_counter()
        t_pgd_inf = end_pgd_inf - start_pgd_inf
        total_time_pgd_inf += t_pgd_inf

        # FGSM metrics
        fgsm_metrics = get_metrics(y_test, y_fgsm_proba, THRESHOLD)
        fgsm_metrics["ASR_I"] = calculate_ASR_I(
            y_test,
            y_clean_proba,
            y_fgsm_proba,
            THRESHOLD,
        )
        fgsm_metrics["time_gen"] = t_fgsm_gen
        fgsm_metrics["time_inf"] = t_fgsm_inf

        # PGD metrics
        pgd_metrics = get_metrics(y_test, y_pgd_proba, THRESHOLD)
        pgd_metrics["ASR_I"] = calculate_ASR_I(
            y_test,
            y_clean_proba,
            y_pgd_proba,
            THRESHOLD,
        )
        pgd_metrics["time_gen"] = t_pgd_gen
        pgd_metrics["time_inf"] = t_pgd_inf

        results.append(
            {
                "epsilon": eps,
                "FGSM": fgsm_metrics,
                "PGD": pgd_metrics,
            }
        )

        del X_fgsm, X_pgd, y_fgsm_proba, y_pgd_proba
        gc.collect()

    accumulated_times = {
        "FGSM_total_gen": total_time_fgsm_gen,
        "FGSM_total_inf": total_time_fgsm_inf,
        "FGSM_total_time": total_time_fgsm_gen + total_time_fgsm_inf,
        "PGD_total_gen": total_time_pgd_gen,
        "PGD_total_inf": total_time_pgd_inf,
        "PGD_total_time": total_time_pgd_gen + total_time_pgd_inf,
    }

    return results, accumulated_times


# ------------------------------------------------------------
# SINGLE-RUN EXECUTION
# ------------------------------------------------------------
def run_single_experiment(X, y, feature_mask_1d, run_id, seed):
    """
    Executes one complete run of the black-box evaluation.
    """
    tf.keras.backend.clear_session()
    gc.collect()
    set_seed(seed)

    (
        X_train_2d,
        X_val_2d,
        X_test_2d,
        y_train_res,
        y_val,
        y_test,
        size,
        scaler,
    ) = prepare_splits(X, y, seed)

    feature_mask_2d = reshape_to_2d(np.array([feature_mask_1d]), size)[0:1]

    # Shadow data are sampled only for surrogate construction.
    # This sampling does not remove samples from target-model training.
    X_shadow, _, _, _ = train_test_split(
        X_train_2d,
        y_train_res,
        train_size=0.20,
        random_state=seed,
        stratify=y_train_res,
    )

    noise = np.random.normal(0, 0.05, X_shadow.shape).astype("float32")
    X_shadow_ood = np.clip(X_shadow + noise, 0.0, 1.0)

    # Target model M1 is trained using the full balanced training set.
    train_ds = make_tf_dataset(
        X_train_2d,
        y_train_res,
        BATCH_SIZE,
        shuffle=True,
        seed=seed,
    )

    val_ds = make_tf_dataset(
        X_val_2d,
        y_val,
        BATCH_SIZE,
        shuffle=False,
        seed=seed,
    )

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-6,
        ),
    ]

    target_model = build_mtl_model_M1(input_shape=(size, size, 1))

    start_train = time.perf_counter()

    history = target_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )

    train_time = time.perf_counter() - start_train
    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)

    # Query the target model to generate pseudo-labels for the shadow dataset.
    # This characterizes a query-based black-box scenario.
    y_shadow_pseudo = target_model.predict(
        X_shadow_ood,
        batch_size=ADV_BATCH_SIZE,
        verbose=1,
    )

    shadow_ds = make_tf_dataset(
        X_shadow_ood,
        y_shadow_pseudo,
        BATCH_SIZE,
        shuffle=True,
        seed=seed,
    )

    surrogate_model = build_surrogate_model(input_shape=(size, size, 1))

    surrogate_callbacks = [
        EarlyStopping(
            monitor="loss",
            patience=5,
            restore_best_weights=True,
        )
    ]

    surrogate_model.fit(
        shadow_ds,
        epochs=30,
        callbacks=surrogate_callbacks,
        verbose=0,
    )

    y_clean_proba = target_model.predict(
        X_test_2d.astype("float32"),
        batch_size=ADV_BATCH_SIZE,
        verbose=0,
    )

    clean_metrics = get_metrics(y_test, y_clean_proba, THRESHOLD)

    y_clean_pred = (y_clean_proba.reshape(-1) > THRESHOLD).astype("int32")
    y_test_flat = np.asarray(y_test).reshape(-1).astype("int32")
    originally_malicious = (y_test_flat == 1) & (y_clean_pred == 1)

    print("\n" + "-" * 60)
    print(f"[Black-Box Diagnostic - Run {run_id}]")
    print("Clean metrics:", clean_metrics)
    print("y_test distribution:", np.bincount(y_test_flat))
    print("Clean prediction distribution:", np.bincount(y_clean_pred))
    print("Correctly detected malicious samples:", originally_malicious.sum())
    print(
        "Clean probabilities - min/max/mean:",
        float(y_clean_proba.min()),
        float(y_clean_proba.max()),
        float(y_clean_proba.mean()),
    )
    print("-" * 60)

    adversarial_metrics, adversarial_times = evaluate_adversarial_blackbox(
        target_model,
        surrogate_model,
        X_test_2d,
        y_test,
        EPSILONS,
        y_clean_proba,
        feature_mask_2d,
    )

    print(f"\n[Run {run_id} Completed] Target Time = {train_time:.2f}s")
    print(f" -> FGSM Total Time (Gen+Inf): {adversarial_times['FGSM_total_time']:.4f}s")
    print(f" -> PGD Total Time (Gen+Inf): {adversarial_times['PGD_total_time']:.4f}s")

    for adv in adversarial_metrics:
        print(
            f"   -> Eps={adv['epsilon']} | "
            f"FGSM (F1={adv['FGSM']['F1']:.4f}, ASR_I={adv['FGSM']['ASR_I']:.4f}) | "
            f"PGD (F1={adv['PGD']['F1']:.4f}, ASR_I={adv['PGD']['ASR_I']:.4f})"
        )

    result = {
        "run": run_id,
        "seed": seed,
        "best_epoch": best_epoch,
        "train_time": train_time,
        "clean": {
            **clean_metrics,
            "n_detected_malicious_clean": int(originally_malicious.sum()),
        },
        "adversarial": adversarial_metrics,
        "adversarial_times": adversarial_times,
    }

    del target_model, surrogate_model, train_ds, val_ds, shadow_ds
    del X_train_2d, X_val_2d, X_test_2d
    del X_shadow, X_shadow_ood
    del y_train_res, y_val, y_test, y_clean_proba

    gc.collect()
    tf.keras.backend.clear_session()

    return result


def _worker_run(run_id, seed, X, y, feature_mask_1d, return_dict):
    """
    Function called by the subprocess.
    """
    configure_gpu()

    result = run_single_experiment(
        X,
        y,
        feature_mask_1d,
        run_id,
        seed,
    )

    return_dict[run_id] = result


# ------------------------------------------------------------
# SUMMARIZATION AND EXPORT
# ------------------------------------------------------------
def summarize_adversarial_results(all_results):
    """
    Prints the mean and standard deviation of adversarial metrics and times.
    """
    print("\n" + "=" * 70)
    print("ADVERSARIAL RESULTS (QUERY-BASED SURROGATE BLACK-BOX): MEAN ± STANDARD DEVIATION")
    print("=" * 70)

    for attack in ["FGSM", "PGD"]:
        print(f"\n--- Attack: {attack} ---")

        for eps in EPSILONS:
            print(f"Epsilon = {eps}")

            for metric in ["Acc", "Prec", "Rec", "F1", "AUC", "ASR_I"]:
                values = [
                    adv_result[attack][metric]
                    for r in all_results
                    for adv_result in r["adversarial"]
                    if adv_result["epsilon"] == eps
                ]
                print(f"  {metric}: {np.mean(values):.4f} ± {np.std(values):.4f}")

    print("\n" + "=" * 70)
    print("EXECUTION TIMES: MEAN ± STANDARD DEVIATION ACROSS RUNS")
    print("=" * 70)

    for attack in ["FGSM", "PGD"]:
        print(f"\n--- Times {attack} ---")

        for eps in EPSILONS:
            gen_times = [
                adv_result[attack]["time_gen"]
                for r in all_results
                for adv_result in r["adversarial"]
                if adv_result["epsilon"] == eps
            ]
            inf_times = [
                adv_result[attack]["time_inf"]
                for r in all_results
                for adv_result in r["adversarial"]
                if adv_result["epsilon"] == eps
            ]

            print(f"Epsilon = {eps}:")
            print(f"  Generation Time: {np.mean(gen_times):.4f}s ± {np.std(gen_times):.4f}s")
            print(f"  Inference Time: {np.mean(inf_times):.4f}s ± {np.std(inf_times):.4f}s")

        total_gen_times = [r["adversarial_times"][f"{attack}_total_gen"] for r in all_results]
        total_inf_times = [r["adversarial_times"][f"{attack}_total_inf"] for r in all_results]
        total_times = [r["adversarial_times"][f"{attack}_total_time"] for r in all_results]

        print(f"\nAccumulated Total Time {attack} (All Epsilons):")
        print(f"  Total Generation: {np.mean(total_gen_times):.4f}s ± {np.std(total_gen_times):.4f}s")
        print(f"  Total Inference: {np.mean(total_inf_times):.4f}s ± {np.std(total_inf_times):.4f}s")
        print(f"  TOTAL (GENERATION + INFERENCE): {np.mean(total_times):.4f}s ± {np.std(total_times):.4f}s")


def export_results_to_csv(
    all_results,
    output_path="adversarial_metrics_query_surrogate_blackbox_HIKARI.csv",
):
    """
    Exports clean and adversarial metrics to CSV, including times.
    """
    rows = []

    for result in all_results:
        run = result["run"]
        seed = result["seed"]
        clean = result.get("clean", {})
        adv_times = result.get("adversarial_times", {})

        for adv in result["adversarial"]:
            eps = adv["epsilon"]

            for attack in ["FGSM", "PGD"]:
                row = {
                    "run": run,
                    "seed": seed,
                    "condition": "adversarial",
                    "scenario": "query_based_surrogate_blackbox",
                    "attack": attack,
                    "epsilon": eps,
                    **adv[attack],
                    "total_time_gen_accumulated": adv_times.get(f"{attack}_total_gen", np.nan),
                    "total_time_inf_accumulated": adv_times.get(f"{attack}_total_inf", np.nan),
                    "total_time_accumulated": adv_times.get(f"{attack}_total_time", np.nan),
                    "clean_Acc": clean.get("Acc", np.nan),
                    "clean_Prec": clean.get("Prec", np.nan),
                    "clean_Rec": clean.get("Rec", np.nan),
                    "clean_F1": clean.get("F1", np.nan),
                    "clean_AUC": clean.get("AUC", np.nan),
                    "n_detected_malicious_clean": clean.get(
                        "n_detected_malicious_clean",
                        np.nan,
                    ),
                    "best_epoch": result["best_epoch"],
                    "train_time": result["train_time"],
                }

                rows.append(row)

    df_results = pd.DataFrame(rows)
    df_results.to_csv(output_path, index=False)

    print(f"\nResults saved to: {output_path}")

    return df_results


# ------------------------------------------------------------
# MAIN FUNCTION
# ------------------------------------------------------------
def main(df_filtrado, feature_mask_1d=None):
    """
    Executes N_RUNS runs of query-based surrogate black-box evaluation.

    The function accepts either:
    - the complete HIKARI dataframe:
      metadata columns + 79 numerical features + traffic_category + Label; or
    - the filtered dataframe:
      79 numerical features + traffic_category + Label.

    traffic_category and Label are response variables. They are not used as
    features and are not included in the plausibility mask.

    If feature_mask_1d is not provided, the HIKARI plausibility mask is built
    automatically from the feature names.
    """
    try:
        mp.set_start_method("spawn")
    except RuntimeError:
        pass

    df = df_filtrado.copy()

    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    df.drop_duplicates(inplace=True)

    X, y, feature_cols = infer_hikari_feature_target_columns(df)

    if feature_mask_1d is None:
        print("\n[Info] No external mask received.")
        print("[Info] Building the HIKARI plausibility mask from feature names.")

        feature_mask_1d, mask_series = build_hikari_plausibility_mask(feature_cols)
    else:
        if isinstance(feature_mask_1d, pd.Series):
            # Align a named mask with the inferred feature order.
            feature_mask_1d = feature_mask_1d.reindex(feature_cols).values

        feature_mask_1d = np.asarray(feature_mask_1d, dtype="float32").reshape(-1)

        if len(feature_mask_1d) != X.shape[1]:
            raise ValueError(
                f"feature_mask_1d has length {len(feature_mask_1d)}, "
                f"but X has {X.shape[1]} features."
            )

        mask_series = pd.Series(feature_mask_1d, index=feature_cols, dtype="float32")

    if len(feature_mask_1d) != X.shape[1]:
        raise ValueError(
            f"feature_mask_1d has length {len(feature_mask_1d)}, "
            f"but X has {X.shape[1]} features."
        )

    summarize_hikari_plausibility_mask(mask_series)

    all_results = []

    manager = mp.Manager()
    return_dict = manager.dict()

    print("\n" + "=" * 50)
    print(f"STARTING {N_RUNS} RUNS (QUERY-BASED SURROGATE BLACK-BOX)")
    print("=" * 50)

    for run in range(N_RUNS):
        seed = 42 + run

        print(
            f"\n[{time.strftime('%H:%M:%S')}] "
            f"Isolated process -> Run {run + 1}/{N_RUNS}"
        )

        p = mp.Process(
            target=_worker_run,
            args=(run + 1, seed, X, y, feature_mask_1d, return_dict),
        )

        p.start()
        p.join()

        if (run + 1) in return_dict:
            all_results.append(return_dict[run + 1])
            del return_dict[run + 1]

    summarize_adversarial_results(all_results)

    df_results = export_results_to_csv(all_results)

    return all_results, df_results

In [ ]:
# Running

import importlib
import ids_engine_blackbox2

# Reload the module in case you have just edited the file
importlib.reload(ids_engine_blackbox2)

# 2. Run the experiment
all_results, df_results = ids_engine_blackbox2.main(df_filtrado)

# 3. View the result
df_results.head()